<a href="https://colab.research.google.com/github/Abhishek-Khelge/inference/blob/main/ProfilingBaseLine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1: installs if needed
!pip -q install transformers accelerate psutil



In [3]:
# Cell 2: imports
import gc
import time
import psutil
import torch
import torch.profiler
from transformers import AutoModelForCausalLM, AutoTokenizer


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


2.10.0+cu128
True
Tesla T4


In [4]:
# Cell 3: model and input setup
model_name = "gpt2-medium"
prompt = "Alan Turing was a pioneering computer scientist whose work laid the foundation for modern computing."
num_new_tokens_to_generate = 50

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
inputs = tokenizer(prompt, return_tensors="pt")

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("pad_token_id:", tokenizer.pad_token_id)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

torch: 2.10.0+cu128
cuda available: True
pad_token_id: 50256


In [5]:
# Cell 4: helper functions
def run_inference(model_to_run, input_data, max_tokens, pad_id):
    with torch.no_grad():
        outputs = model_to_run.generate(
            input_ids=input_data["input_ids"],
            attention_mask=input_data.get("attention_mask"),
            max_new_tokens=max_tokens,
            pad_token_id=pad_id,
        )
    return outputs

def get_memory_usage_gb(model_to_measure, device_type="cpu"):
    gc.collect()

    if device_type == "cuda" and torch.cuda.is_available():
        torch.cuda.empty_cache()
        mem_allocated_bytes = torch.cuda.memory_allocated()
        return mem_allocated_bytes / (1024**3)

    if device_type == "cpu":
        model_mem_bytes = sum(
            p.numel() * p.element_size() for p in model_to_measure.parameters()
        )
        return model_mem_bytes / (1024**3)

    return 0

def get_process_memory_gb():
    gc.collect()
    return psutil.Process().memory_info().rss / (1024**3)


In [6]:
# Cell 5: CPU profiling
model.to("cpu")
inputs_cpu = {k: v.to("cpu") for k, v in inputs.items()}
cpu_model_load_mem_allocated_gb = get_memory_usage_gb(model, device_type="cpu")

start_time_cpu_wall = time.time()
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU]
) as prof_cpu:
    generated_outputs_cpu = run_inference(
        model,
        inputs_cpu,
        num_new_tokens_to_generate,
        tokenizer.pad_token_id,
    )
end_time_cpu_wall = time.time()

cpu_e2e_latency_s = end_time_cpu_wall - start_time_cpu_wall
cpu_generated_tokens = generated_outputs_cpu.shape[-1] - inputs_cpu["input_ids"].shape[-1]
cpu_tokens_per_second = cpu_generated_tokens / cpu_e2e_latency_s

print("--- CPU Metrics ---")
print(f"Model memory: {cpu_model_load_mem_allocated_gb:.4f} GB")
print(f"Process RSS: {get_process_memory_gb():.4f} GB")
print(f"E2E Latency: {cpu_e2e_latency_s:.4f} s")
print(f"Tokens/sec: {cpu_tokens_per_second:.2f} tokens/s")

print("\nGenerated text:")
print(tokenizer.decode(generated_outputs_cpu[0], skip_special_tokens=True))


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


--- CPU Metrics ---
Model memory: 1.3218 GB
Process RSS: 2.6163 GB
E2E Latency: 23.6382 s
Tokens/sec: 2.12 tokens/s

Generated text:
Alan Turing was a pioneering computer scientist whose work laid the foundation for modern computing. He was born in London on August 10, 1928. He was the son of a German immigrant father and a British mother. Turing was educated at Cambridge University and the University of Cambridge, where he received his doctorate in mathematics in 1936. He was


In [7]:
# Cell 6: CPU profiler summary
print(
    prof_cpu.key_averages().table(
        sort_by="cpu_time_total",
        row_limit=15
    )
)


-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                 Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          aten::addmm        73.66%       13.377s        74.85%       13.594s       2.832ms          4800  
                                         aten::linear         0.01%       2.105ms        11.05%        2.007s      40.150ms            50  
                                         aten::matmul         0.01%       2.352ms        11.04%        2.005s      40.093ms            50  
                                             aten::mm        11.02%        2.001s        11.02%        2.001s      40.026ms            50  
                    

In [8]:
# Cell 7: GPU profiling, only if Colab GPU runtime is enabled
if torch.cuda.is_available():
    model.to("cuda")
    inputs_gpu = {k: v.to("cuda") for k, v in inputs.items()}
    gpu_model_load_mem_allocated_gb = get_memory_usage_gb(model, device_type="cuda")

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    with torch.profiler.profile(
        activities=[
            torch.profiler.ProfilerActivity.CPU,
            torch.profiler.ProfilerActivity.CUDA,
        ]
    ) as prof_gpu:
        start_event.record()
        generated_outputs_gpu = run_inference(
            model,
            inputs_gpu,
            num_new_tokens_to_generate,
            tokenizer.pad_token_id,
        )
        end_event.record()
        torch.cuda.synchronize()

    gpu_e2e_latency_s_event = start_event.elapsed_time(end_event) / 1000.0
    gpu_generated_tokens = generated_outputs_gpu.shape[-1] - inputs_gpu["input_ids"].shape[-1]
    gpu_tokens_per_second = gpu_generated_tokens / gpu_e2e_latency_s_event
    gpu_peak_mem_gb = torch.cuda.max_memory_allocated() / (1024**3)

    print("--- GPU Metrics ---")
    print(f"Initial allocated GPU memory: {gpu_model_load_mem_allocated_gb:.4f} GB")
    print(f"Peak GPU memory: {gpu_peak_mem_gb:.4f} GB")
    print(f"E2E Latency (CUDA Event): {gpu_e2e_latency_s_event:.4f} s")
    print(f"Tokens/sec: {gpu_tokens_per_second:.2f} tokens/s")

    print("\nGenerated text:")
    print(tokenizer.decode(generated_outputs_gpu[0], skip_special_tokens=True))
else:
    print("CUDA is not available. Enable GPU in Colab from Runtime > Change runtime type.")


--- GPU Metrics ---
Initial allocated GPU memory: 1.3453 GB
Peak GPU memory: 1.3666 GB
E2E Latency (CUDA Event): 2.5550 s
Tokens/sec: 19.57 tokens/s

Generated text:
Alan Turing was a pioneering computer scientist whose work laid the foundation for modern computing. He was born in London on August 10, 1928. He was the son of a German immigrant father and a British mother. Turing was educated at Cambridge University and the University of Cambridge, where he received his doctorate in mathematics in 1936. He was


In [9]:
# Cell 8: GPU profiler summary
if torch.cuda.is_available():
    print(
        prof_gpu.key_averages().table(
            sort_by="cuda_time_total",
            row_limit=15
        )
    )


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                            aten::addmm        13.44%     274.158ms        26.68%     544.084ms     113.351us     289.572ms        63.12%     290.999ms      60.625us          4800  
std::enable_if<!(false), void>::type internal::gemvx...         0.00%       0.000us         0.00%       0.000us       0.000us     230.806ms        50.31%     230.806ms      65.421us          3528  
         

In [10]:
# Cell 9: final comparison
print("--- Final Comparison ---")
print(f"CPU latency: {cpu_e2e_latency_s:.4f} s")
print(f"CPU tokens/sec: {cpu_tokens_per_second:.2f}")

if torch.cuda.is_available():
    print(f"GPU latency: {gpu_e2e_latency_s_event:.4f} s")
    print(f"GPU tokens/sec: {gpu_tokens_per_second:.2f}")
    print(f"GPU peak memory: {gpu_peak_mem_gb:.4f} GB")


--- Final Comparison ---
CPU latency: 23.6382 s
CPU tokens/sec: 2.12
GPU latency: 2.5550 s
GPU tokens/sec: 19.57
GPU peak memory: 1.3666 GB
